# Getting Started with PySpark and Pandas

**Estimated time:** 60 minutes | **Author:** Ritika Joshi

---

## Table of Contents

1. [PySpark Overview](#1-pyspark-overview)
2. [Understanding Pandas](#2-understanding-pandas)
3. [Setting Up the Environment](#3-setting-up-the-environment)
4. [Initializing a Spark Session](#4-initializing-a-spark-session)
5. [Loading Data into Pandas](#5-loading-data-into-pandas)
6. [Exploring the Dataset](#6-exploring-the-dataset)
7. [Converting a Pandas DataFrame to Spark](#7-converting-a-pandas-dataframe-to-spark)
8. [Inspecting the Spark DataFrame Schema](#8-inspecting-the-spark-dataframe-schema)
9. [Basic Data Exploration](#9-basic-data-exploration)
10. [Working with Columns](#10-working-with-columns)
11. [Grouping and Summarizing](#11-grouping-and-summarizing)
12. [User-Defined Functions (UDFs)](#12-user-defined-functions-udfs)
13. [Querying with Spark SQL](#13-querying-with-spark-sql)
14. [Running SQL Queries](#14-running-sql-queries)


## Introduction

**PySpark** is the Python API for Apache Spark — a distributed computing system built for large-scale data processing. By distributing computation across multiple nodes, PySpark can handle datasets reaching terabytes or even petabytes, making it ideal for Big Data workloads.

**Pandas** is a Python library optimised for data manipulation and analysis on structured data. It provides expressive, high-level data structures (Series and DataFrame) and excels at tasks such as data cleaning, transformation, aggregation, and exploratory analysis when the dataset fits in memory.

In this notebook, we use a sample COVID-19 dataset — containing cumulative cases, deaths, and vaccinations by continent — to demonstrate how both tools can be used together in a practical workflow.

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain the core use cases of PySpark (distributed computing) and Pandas (in-memory analysis).
- Install and configure both libraries in a Python environment.
- Load data into Pandas and PySpark DataFrames and perform initial exploration.
- Convert a Pandas DataFrame to a Spark DataFrame for distributed processing.
- Create derived columns, filter records, and aggregate data using PySpark.
- Execute SQL queries on Spark DataFrames using Spark SQL and user-defined functions (UDFs).


---

## 1. PySpark Overview

PySpark is the Python API for Apache Spark, designed for large-scale data processing across distributed clusters. It provides abstractions for both low-level RDDs and high-level DataFrames, enabling fault-tolerant, parallel computation.

| Aspect | Details |
|---|---|
| **Distributed computing** | Processes data across multiple cluster nodes simultaneously |
| **High performance** | In-memory computation outperforms traditional MapReduce frameworks |
| **Big data support** | Handles datasets larger than the memory of any single machine |
| **Python ecosystem** | Integrates natively with Pandas, NumPy, Matplotlib, and more |

**Typical use cases:** large-scale ETL pipelines, distributed machine learning, real-time stream processing, and complex analytical queries on multi-terabyte datasets.

**Limitations:** requires more complex cluster setup and configuration; steeper learning curve than single-machine tools.


---

## 2. Understanding Pandas

Pandas is a Python library for data manipulation and analysis. Its two primary data structures — **Series** (one-dimensional) and **DataFrame** (two-dimensional) — make it straightforward to load, clean, transform, and analyse structured data.

| Aspect | Details |
|---|---|
| **Data I/O** | Reads and writes CSV, Excel, JSON, SQL, Parquet, and more |
| **Data cleaning** | Built-in tools for handling missing values and duplicates |
| **Data analysis** | Rich statistical and aggregation functions |
| **Visualisation** | Integrates with Matplotlib and Seaborn for quick plots |

**Typical use cases:** exploratory data analysis, feature engineering, data wrangling, and rapid prototyping with medium-sized datasets.

**Limitations:** single-machine, in-memory only; not suited for datasets that exceed available RAM.


---

## 3. Setting Up the Environment

Install the required libraries using pip. If a package is already present, pip will skip its installation.


In [ ]:
# Install required packages
!pip install pyspark findspark pandas --quiet


---

## 4. Initializing a Spark Session

A `SparkSession` is the single entry point for all PySpark functionality. The `.getOrCreate()` pattern ensures that only one session is active at a time — useful when re-running cells in a notebook.

- **`findspark`** locates the local Spark installation so Python can import it.
- **Arrow optimisation** is enabled to accelerate Pandas ↔ Spark conversions.


In [ ]:
import findspark
findspark.init()  # Locate the local Spark installation

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DateType
import pandas as pd

# Create (or retrieve) a Spark session
spark = (
    SparkSession.builder
    .appName("COVID-19 Data Analysis")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

# Confirm the session is active
if isinstance(spark, SparkSession):
    print("SparkSession is active and ready.")
else:
    print("SparkSession could not be created.")


---

## 5. Loading Data into Pandas

We load the COVID-19 dataset directly from a remote URL into a Pandas DataFrame using `pd.read_csv()`. The dataset is sourced from *Our World in Data* and contains the latest available figures per country.


In [ ]:
DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud"
    "/KpHDlIzdtR63BdTofl1mOg/owid-covid-latest.csv"
)

vaccination_data = pd.read_csv(DATA_URL)
print(f"Dataset loaded: {vaccination_data.shape[0]:,} rows x {vaccination_data.shape[1]} columns")


---

## 6. Exploring the Dataset

Before any transformation, it is good practice to inspect the raw data. We focus on five columns used throughout this notebook:

| Column | Description |
|---|---|
| `continent` | Geographic region |
| `total_cases` | Cumulative confirmed COVID-19 cases |
| `total_deaths` | Cumulative confirmed deaths |
| `total_vaccinations` | Total vaccine doses administered |
| `population` | Total population |

> **Note:** `fillna(0)` replaces `NaN` values with `0` so that numeric columns can be safely cast to integer types for Spark.


In [ ]:
COLUMNS = ['continent', 'total_cases', 'total_deaths', 'total_vaccinations', 'population']

print("First 5 records (selected columns):")
print(vaccination_data[COLUMNS].head())


---

## 7. Converting a Pandas DataFrame to Spark

To leverage Spark's distributed processing, we convert the Pandas DataFrame to a Spark DataFrame. This requires:

1. **Defining an explicit schema** so Spark knows each column's name and data type.
2. **Cleaning data types** to ensure compatibility — `continent` as `str`, numeric columns as `int64` (mapped to Spark's `LongType`).

| Spark type | Python / Pandas equivalent |
|---|---|
| `StringType` | `str` |
| `LongType` | `int64` |

Using `fillna(0)` before casting prevents `NaN` from raising a `TypeError` during conversion.


In [ ]:
# Define the Spark schema
schema = StructType([
    StructField("continent",          StringType(), nullable=True),
    StructField("total_cases",        LongType(),   nullable=True),
    StructField("total_deaths",       LongType(),   nullable=True),
    StructField("total_vaccinations", LongType(),   nullable=True),
    StructField("population",         LongType(),   nullable=True),
])

# Align Pandas column types with the Spark schema
vaccination_data["continent"]          = vaccination_data["continent"].astype(str)
vaccination_data["total_cases"]        = vaccination_data["total_cases"].fillna(0).astype("int64")
vaccination_data["total_deaths"]       = vaccination_data["total_deaths"].fillna(0).astype("int64")
vaccination_data["total_vaccinations"] = vaccination_data["total_vaccinations"].fillna(0).astype("int64")
vaccination_data["population"]         = vaccination_data["population"].fillna(0).astype("int64")

# Create the Spark DataFrame using only the schema fields
spark_df = spark.createDataFrame(vaccination_data[schema.fieldNames()])

print("Spark DataFrame created successfully.")
spark_df.show(5)


---

## 8. Inspecting the Spark DataFrame Schema

`printSchema()` displays each column's name, data type, and nullability in a tree format. Verifying the schema immediately after creation helps catch type mismatches before they surface as runtime errors later in the pipeline.


In [ ]:
print("Spark DataFrame schema:")
spark_df.printSchema()


---

## 9. Basic Data Exploration

With the Spark DataFrame ready, we perform three foundational exploration steps: viewing sample rows, projecting a column subset, and filtering rows by a condition.

### 9.1 Viewing a Sample of Records

`select()` restricts the output to the specified columns; `show(n)` prints the first *n* rows.


In [ ]:
COLUMNS = ['continent', 'total_cases', 'total_deaths', 'total_vaccinations', 'population']

spark_df.select(COLUMNS).show(5)


### 9.2 Selecting Specific Columns

We can project any subset of columns. Here we display only `continent` and `total_cases` to keep the output concise.


In [ ]:
print("Columns: continent and total_cases")
spark_df.select("continent", "total_cases").show(5)


### 9.3 Filtering Records

`filter()` (or equivalently `where()`) applies a row-level predicate. The result contains only the rows that satisfy the condition.


In [ ]:
print("Records where total_cases > 1,000,000:")
spark_df.filter(spark_df['total_cases'] > 1_000_000).show(5)


---

## 10. Working with Columns

`withColumn()` adds or replaces a column without mutating the original DataFrame (Spark DataFrames are immutable).

We derive a `death_percentage` column:

$$\text{death\_percentage} = \frac{\text{total\_deaths}}{\text{population}} \times 100$$

We then format the value to two decimal places and append a `%` symbol using `F.format_number()`, `F.concat()`, and `F.lit()`.


In [ ]:
from pyspark.sql import functions as F

# Compute raw death percentage
spark_df_with_pct = spark_df.withColumn(
    "death_percentage",
    (spark_df["total_deaths"] / spark_df["population"]) * 100
)

# Format to 2 decimal places and append '%'
spark_df_with_pct = spark_df_with_pct.withColumn(
    "death_percentage",
    F.concat(
        F.format_number(spark_df_with_pct["death_percentage"], 2),
        F.lit("%")
    )
)

DISPLAY_COLS = ["continent", "total_cases", "total_deaths",
                "population", "total_vaccinations", "death_percentage"]
spark_df_with_pct.select(DISPLAY_COLS).show(5)


---

## 11. Grouping and Summarizing

`groupBy()` partitions the DataFrame by one or more columns; `agg()` applies an aggregation function to each partition.

Here we sum confirmed deaths by continent to produce a high-level regional overview. Results are sorted in descending order for readability.


In [ ]:
print("Total deaths per continent (descending):")
(
    spark_df
    .groupBy("continent")
    .agg(F.sum("total_deaths").alias("total_deaths_sum"))
    .orderBy("total_deaths_sum", ascending=False)
    .show()
)


---

## 12. User-Defined Functions (UDFs)

A **UDF** (User-Defined Function) lets you apply arbitrary Python logic to a Spark column. After registration with `spark.udf.register()`, the function is available in both the DataFrame API and Spark SQL.

> **Performance note:** Native Spark functions (e.g. `F.col`, `F.when`) avoid Python serialisation overhead and are generally faster than UDFs. Prefer UDFs only when no built-in equivalent exists.

The UDF below doubles `total_deaths` — a simple placeholder demonstrating the registration pattern.


In [ ]:
from pyspark.sql.types import IntegerType

def convert_total_deaths(total_deaths: int) -> int:
    """Return double the total_deaths value."""
    return total_deaths * 2

# Register the UDF for use in Spark SQL
spark.udf.register("convert_total_deaths", convert_total_deaths, IntegerType())
print("UDF 'convert_total_deaths' registered successfully.")


---

## 13. Querying with Spark SQL

Spark SQL allows you to write standard SQL against any DataFrame registered as a **temporary view**. This is particularly useful for analysts comfortable with SQL who prefer not to learn the DataFrame API.

We:
1. Register `spark_df` as the temporary view `data_v`.
2. Run a SQL query that calls our `convert_total_deaths` UDF alongside the raw column.


In [ ]:
# Register the Spark DataFrame as a temporary SQL view
spark.sql("DROP VIEW IF EXISTS data_v")
spark_df.createTempView('data_v')

# Query the view using Spark SQL and the registered UDF
result = spark.sql(
    'SELECT continent, total_deaths,'
    ' convert_total_deaths(total_deaths) AS doubled_total_deaths'
    ' FROM data_v'
)
result.show()


---

## 14. Running SQL Queries

With `data_v` registered, we can run arbitrary SQL queries against the Spark DataFrame.

### 14.1 Retrieving All Records

`SELECT *` returns every column from the view. For large tables, narrow the column list to avoid unnecessary data transfer.


In [ ]:
spark.sql('SELECT * FROM data_v').show()


### 14.2 Filtering by Vaccination Totals

The query below returns only the continents where `total_vaccinations` exceed one million.


In [ ]:
print("Continents with more than 1,000,000 total vaccinations:")
spark.sql(
    'SELECT continent FROM data_v WHERE total_vaccinations > 1000000'
).show()


---

## Conclusion

In this notebook, we covered an end-to-end workflow combining Pandas and PySpark:

| Step | Tool | Key operation |
|---|---|---|
| Load data | Pandas | `pd.read_csv()` |
| Explore & clean | Pandas | `.head()`, `.fillna()`, `.astype()` |
| Convert to Spark | PySpark | `spark.createDataFrame()` |
| Inspect schema | PySpark | `printSchema()` |
| Explore at scale | PySpark | `.select()`, `.filter()`, `.show()` |
| Derive columns | PySpark | `withColumn()`, `F.concat()` |
| Aggregate | PySpark | `groupBy()`, `agg()` |
| Custom logic | PySpark UDF | `spark.udf.register()` |
| SQL queries | Spark SQL | `createTempView()`, `spark.sql()` |

**Key takeaway:** Pandas and PySpark are complementary tools. Use Pandas for fast, interactive analysis on datasets that fit comfortably in memory; reach for PySpark when data scale, fault-tolerance, or distributed computation is required.

---

*Author: Ritika Joshi | First published: 2024-09-23*
